# TP NLP – Prétraitement du Texte
## Pipeline complet en Python

Corpus médical avec signalements libres.

In [6]:


import nltk

# Télécharger ressources NLTK
nltk.download('punkt')
nltk.download('stopwords')

ModuleNotFoundError: No module named 'nltk'

In [ ]:
# Importations
import re
import unicodedata
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

# Chargement du modèle spaCy français
nlp = spacy.load("fr_core_news_sm")

# Corpus original
corpus = [
    "Le Patient a DES Fievres Elevees depuis 3 jours!!! et tousse bcp",
    "pas de symptomes graves... juste une legere FATIGUE & maux de tete",
    "URGENT : douleurs thoraciques intenses + essoufflement depuis hier soir",
    "L’enfant ne mange PAS et pleure tt la nuit, fievre a 39C",
    "Amelioration notable apres le tt - patient stabilise :)",
    "Toux persistante depuis + de 2 semaines, aucune amelioration malgre les medocs",
    "RAS rien de particulier signaler aujourd’hui le patient va BIEN"
]

## Exercice 1 – Analyse préliminaire

In [ ]:
# Q1 : Types de bruits pour chaque phrase
bruits = []
for i, phrase in enumerate(corpus, 1):
    types = []
    if any(c.isupper() for c in phrase if c.isalpha()):
        types.append("majuscules")
    if any(c in "!?.,;:+-&%*()[]" for c in phrase):
        types.append("ponctuation/caractères spéciaux")
    if any(c in ":)" for c in phrase):
        types.append("emoji")
    if re.search(r'\b(tt|bcp|RAS)\b', phrase):
        types.append("abréviations")
    if any(ord(c) > 127 for c in phrase):   # accents, etc.
        types.append("accents")
    if '  ' in phrase:
        types.append("espaces multiples")
    print(f"Phrase {i}: {types if types else 'aucun bruit'}")
    bruits.append(types)

In [ ]:
# Q2 : Statistiques générales
nb_phrases = len(corpus)
nb_mots_naif = sum(len(phrase.split()) for phrase in corpus)
longueur_moyenne = nb_mots_naif / nb_phrases
print(f"Nombre total de phrases : {nb_phrases}")
print(f"Nombre total de mots (séparation naïve) : {nb_mots_naif}")
print(f"Longueur moyenne d'une phrase en mots : {longueur_moyenne:.2f}")

**Q3** : La phrase la plus problématique est la quatrième : `"L’enfant ne mange PAS et pleure tt la nuit, fievre a 39C"`. Elle contient des majuscules (`PAS`), une abréviation (`tt`), une apostrophe, une virgule, un chiffre avec lettre (`39C`) et des accents sur `fievre`. Sans prétraitement, un modèle aurait du mal à reconnaître `tt` comme `traitement` et ne grouperait pas `39C` avec la fièvre.

## Exercice 2 – Normalisation du texte

In [ ]:
# Q1 : Mise en minuscules + suppression des accents
def normaliser_casse(texte):
    texte = texte.lower()
    texte = unicodedata.normalize('NFKD', texte).encode('ASCII', 'ignore').decode('utf-8')
    return texte

# Test
print(normaliser_casse(corpus[0]))

In [ ]:
# Q2 : Suppression ponctuation + caractères spéciaux
# Justification : les chiffres (température 39, 2 jours) sont conservés car informations cliniques.
def supprimer_ponctuation(texte):
    ponctuation = r"[!?.,;:+-/#$%&*()\[\]{}<>]"
    return re.sub(ponctuation, ' ', texte)  # remplace par espace

# Test
print(supprimer_ponctuation(normaliser_casse(corpus[0])))

In [ ]:
# Q3 : Nettoyage des espaces multiples et tabulations
def nettoyer_espaces(texte):
    texte = re.sub(r'\s+', ' ', texte)  # remplace tabulations, retours chariot, etc.
    return texte.strip()

# Avant/après pour phrase #5 (indice 4)
print("Avant :", repr(corpus[4]))
print("Après :", repr(nettoyer_espaces(corpus[4])))

In [ ]:
# Q4 : Expansion des abréviations médicales
abrev_dict = {
    r'\btt\b': 'traitement',
    r'\bbcp\b': 'beaucoup',
    r'\bras\b': 'rien a signaler'
}

def expand_abrev(texte, dico):
    for pattern, repl in dico.items():
        texte = re.sub(pattern, repl, texte, flags=re.IGNORECASE)
    return texte

# Test
print(expand_abrev(corpus[4], abrev_dict))

In [ ]:
# Q5 : Fonction de normalisation complète
def normaliser(texte):
    texte = normaliser_casse(texte)
    texte = supprimer_ponctuation(texte)
    texte = expand_abrev(texte, abrev_dict)
    texte = nettoyer_espaces(texte)
    return texte

corpus_normalise = [normaliser(phrase) for phrase in corpus]
for i, phrase in enumerate(corpus_normalise, 1):
    print(f"Phrase {i}: {phrase}")

## Exercice 3 – Tokenisation

In [ ]:
# Q1 : word_tokenize sur corpus normalisé (avant/après normalisation pour phrase #3)
phrase_3_original = corpus[2]
phrase_3_norm = corpus_normalise[2]
print("Avant normalisation :", word_tokenize(phrase_3_original, language='french'))
print("Après normalisation  :", word_tokenize(phrase_3_norm, language='french'))

In [ ]:
# Q2 : Comparaison des méthodes sur phrase #4
phrase_4 = corpus_normalise[3]  # "lenfant ne mange pas et pleure traitement la nuit fievre a 39c"

# Par mots (word_tokenize)
tokens_mots = word_tokenize(phrase_4, language='french')

# Par sous-mots (WordPiece simulé avec une simple séparation de préfixes fréquents)
# On utilise une approche illustrative : découpage sur les espaces puis sur 'c' pour 39c
tokens_wordpiece = []
for mot in tokens_mots:
    if mot == '39c':
        tokens_wordpiece.extend(['39', 'c'])
    else:
        tokens_wordpiece.append(mot)

# Par caractères
tokens_caracteres = list(phrase_4)

print("Tokens par mots :", tokens_mots)
print("Tokens par sous-mots (WordPiece-like) :", tokens_wordpiece)
print("Tokens par caractères :", tokens_caracteres)

## Exercice 4 – Suppression des stop-words

In [ ]:
# Q1 : Filtrage avec stop-words NLTK français
stop_fr = set(stopwords.words('french'))

tokens_1 = word_tokenize(corpus_normalise[0], language='french')
tokens_1_filtres = [t for t in tokens_1 if t not in stop_fr]
tokens_4 = word_tokenize(corpus_normalise[3], language='french')
tokens_4_filtres = [t for t in tokens_4 if t not in stop_fr]

print("Phrase #1 après stop-words :", tokens_1_filtres)
print("Phrase #4 après stop-words :", tokens_4_filtres)

In [ ]:
# Q2 : Analyse des mots importants supprimés
# a) Phrase #2 : "pas de symptomes graves..." -> après normalisation "pas de symptomes graves"
#    Stop-words suppriment 'pas' (négation) et 'de'. 'pas' est crucial car il indique l'absence de symptômes graves.
# b) Phrase #4 : "ne mange pas" -> 'ne' et 'pas' sont supprimés, on perd la négation "ne mange pas"
#    Ceci est critique : un patient qui ne mange pas indique une anorexie ou un trouble grave.
print("Réponse Q2a : Le mot 'pas' (négation) est supprimé de la phrase #2. Impact : on perd le caractère négatif du signalement.")
print("Réponse Q2b : Le mot 'pas' (négation) et 'ne' sont supprimés de la phrase #4. La négation 'ne mange pas' devient 'mange', inversant le sens clinique.")

In [ ]:
# Q3 : Stop-words personnalisés (conservation des négations et intensités)
stop_perso = stop_fr - {'pas', 'aucune', 'sans', 'tres', 'intense', 'grave'}
tokens_2_orig = word_tokenize(corpus_normalise[1], language='french')
tokens_2_perso = [t for t in tokens_2_orig if t not in stop_perso]
tokens_4_orig = word_tokenize(corpus_normalise[3], language='french')
tokens_4_perso = [t for t in tokens_4_orig if t not in stop_perso]

print("Avec stop-words standard (phrase #2) :", [t for t in tokens_2_orig if t not in stop_fr])
print("Avec stop-words personnalisé (phrase #2) :", tokens_2_perso)
print("\nAvec stop-words standard (phrase #4) :", [t for t in tokens_4_orig if t not in stop_fr])
print("Avec stop-words personnalisé (phrase #4) :", tokens_4_perso)

## Exercice 5 – Stemming et Lemmatisation

In [ ]:
# Q1 : Stemming Snowball (français)
stemmer = SnowballStemmer('french')
mots_test = ['fievres', 'douleurs', 'thoraciques', 'amelioration', 'essoufflement', 'stabilise']
print("Mot original -> Stemming")
for mot in mots_test:
    stem = stemmer.stem(mot)
    valide = "oui" if len(stem) > 2 else "non"
    print(f"{mot:15} -> {stem:15} (valide ? {valide})")

In [ ]:
# Q2 : Lemmatisation spaCy
print("\nMot original -> Lemme")
for mot in mots_test:
    doc = nlp(mot)
    lemme = doc[0].lemma_
    valide = "oui" if lemme in ['fievre', 'douleur', 'thoracique', 'amelioration', 'essoufflement', 'stabiliser'] else "non"
    print(f"{mot:15} -> {lemme:15} (valide ? {valide})")

**Q3** : Dans un contexte médical, la lemmatisation est préférable car elle conserve le sens grammatical (ex. `stabilisé` → `stabiliser`) alors que le stemming produit des racines arbitraires (ex. `stabilis`). Les variations flexionnelles (`fievres` → `fievre`) sont importantes pour ne pas confondre avec d'autres termes, et la lemmatisation garantit un retour au dictionnaire, évitant des ambiguïtés sémantiques.

In [ ]:
# Q4 : Pipeline NLP complet
def pipeline_nlp(corpus_brut):
    resultats = []
    for phrase in corpus_brut:
        # Normalisation
        phrase_norm = normaliser(phrase)
        # Tokenisation
        tokens = word_tokenize(phrase_norm, language='french')
        # Suppression stop-words personnalisée (gardant négations)
        tokens_filtres = [t for t in tokens if t not in stop_perso]
        # Lemmatisation avec spaCy
        doc = nlp(' '.join(tokens_filtres))
        lemmes = [token.lemma_ for token in doc]
        resultats.append(lemmes)
    return resultats

corpus_final = pipeline_nlp(corpus)
for i, lemmes in enumerate(corpus_final, 1):
    print(f"Phrase {i}: {lemmes}")

## Bonus – Synthèse (TF-IDF et gestion des négations)

In [ ]:
# B1 : TF-IDF sur corpus prétraité (sous forme de chaînes)
corpus_str = [' '.join(doc) for doc in corpus_final]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus_str)
scores = tfidf_matrix.sum(axis=0).A1
termes = vectorizer.get_feature_names_out()
top5 = sorted(zip(termes, scores), key=lambda x: x[1], reverse=True)[:5]
print("5 termes les plus discriminants (score TF-IDF cumulé) :")
for terme, score in top5:
    print(f"  {terme}: {score:.3f}")

In [ ]:
# B2 : Comparaison du nombre de features avant/après prétraitement
# Avant prétraitement : tokenisation naive sans normalisation
corpus_brut_str = [phrase.lower() for phrase in corpus]
vectorizer_brut = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b')
tfidf_brut = vectorizer_brut.fit_transform(corpus_brut_str)
features_brut = len(vectorizer_brut.get_feature_names_out())
features_propre = len(vectorizer.get_feature_names_out())
print(f"Nombre de features (mots distincts) avant prétraitement : {features_brut}")
print(f"Nombre de features après prétraitement complet : {features_propre}")
print("=> Le prétraitement réduit le vocabulaire (regroupement, correction), ce qui limite la dimensionnalité et évite le surapprentissage.")

In [ ]:
# B3 : Préservation des négations
# Approche : remplacer "ne ... pas" par un token spécial avant suppression des stop-words
def normaliser_avec_negation(texte):
    texte = normaliser_casse(texte)
    # Détecter les négations 'ne ... pas' ou 'ne mange pas' -> 'negation'
    texte = re.sub(r'\bne\b\s+(\w+)\s+pas\b', r'neg_\1', texte)
    texte = re.sub(r'\bpas\b', '', texte)  # enlève 'pas' seul
    texte = supprimer_ponctuation(texte)
    texte = expand_abrev(texte, abrev_dict)
    texte = nettoyer_espaces(texte)
    return texte

# Test sur phrase #4
phrase4_neg = normaliser_avec_negation(corpus[3])
print("Phrase #4 avec négation préservée :", phrase4_neg)